# Demo: ForecastingAssistant with LLM

This notebook demonstrates the LLM-powered features of `ForecastingAssistant`.
While Tier 0 methods (`profile`, `recommend`, `generate_code`, `forecast`) work
without any LLM, the `ask()` and `explain()` methods leverage a language model
to translate natural-language questions into deterministic forecasting pipelines
and to explain plans in plain language.

**Requirements:**

- Install with LLM extras: `pip install -e ".[llm]"`
- For Ollama: have `ollama serve` running locally
- For cloud providers (OpenAI, Anthropic, etc.): set the corresponding API key env var

**Supported provider strings:**

| Provider | Format | Example |
|---|---|---|
| OpenAI | `openai:model` | `openai:gpt-4o-mini` |
| Anthropic | `anthropic:model` | `anthropic:claude-sonnet-4-20250514` |
| Google | `google:model` | `google:gemini-2.0-flash` |
| Groq | `groq:model` | `groq:llama-3.1-70b-versatile` |
| Mistral | `mistral:model` | `mistral:mistral-large-latest` |
| Ollama | `ollama:model` | `ollama:qwen2.5:7b-instruct` |

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant

## 2. Create sample data

In [2]:
np.random.seed(42)
dates = pd.date_range("2022-01-01", periods=365, freq="D")
trend = np.linspace(100, 200, 365)
seasonality = 20 * np.sin(2 * np.pi * np.arange(365) / 7)  # weekly pattern
noise = np.random.normal(0, 5, 365)

df = pd.DataFrame({
    "date": dates,
    "sales": trend + seasonality + noise,
    "promo": np.random.choice([0.0, 1.0], size=365),
    "temperature": 15 + 10 * np.sin(2 * np.pi * np.arange(365) / 365) + np.random.normal(0, 2, 365),
})

df.head()

,date,sales,promo,temperature
0,2022-01-01,102.483571,1.0,14.197559
1,2022-01-02,115.220033,0.0,18.297761
2,2022-01-03,123.286451,1.0,16.455497
3,2022-01-04,117.117000,1.0,19.685151
4,2022-01-05,91.250459,1.0,18.469831


## 3. Initialize the assistant with an LLM

Choose your provider. Uncomment the one you want to use:

In [ ]:
# --- Option A: Ollama (local, free) ---
# Make sure Ollama is running: `ollama serve`
# and you have pulled the model: `ollama pull qwen2.5:7b-instruct`
# Note: qwen3 has compatibility issues with Ollama's tool calling API.
#       Use qwen2.5:7b-instruct for reliable structured output.
assistant = ForecastingAssistant(llm="ollama:qwen2.5:7b-instruct")

# --- Option B: OpenAI ---
# Requires OPENAI_API_KEY env var
# assistant = ForecastingAssistant(llm="openai:gpt-4o-mini")

# --- Option C: Anthropic ---
# Requires ANTHROPIC_API_KEY env var
# assistant = ForecastingAssistant(llm="anthropic:claude-sonnet-4-20250514")

# --- Option D: Google ---
# Requires GOOGLE_API_KEY env var
# assistant = ForecastingAssistant(llm="google:gemini-2.0-flash")

print(f"LLM: {assistant.llm}")

LLM: ollama:qwen3:8b


## 4. `ask()` — Natural-language questions

The `ask()` method sends a free-form question to the LLM agent. The agent
does NOT make forecasting decisions — it translates your question into calls
to the deterministic engine (`profile_data`, `recommend`, `generate_code_tool`)
and explains the result.

### 4.1 Simple question (no data)

In [4]:
result = assistant.ask(
    question="What forecasting model should I use for a dataset with weekly seasonality and 500 daily observations?"
)

print(result.explanation)

[LLM unavailable] status_code: 400, model_name: qwen3:8b, body: {'message': 'invalid message content type: <nil>', 'type': 'invalid_request_error', 'param': None, 'code': None}. Provide data to get a deterministic recommendation.


/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_82258/2461738756.py:1: UserWarning: LLM call failed (status_code: 400, model_name: qwen3:8b, body: {'message': 'invalid message content type: <nil>', 'type': 'invalid_request_error', 'param': None, 'code': None}), falling back to deterministic mode.
  result = assistant.ask(


In [5]:
# The AskResult may contain a ForecastPlan if the agent produced one
if result.plan:
    print(f"Forecaster:  {result.plan.forecaster}")
    print(f"Estimator:   {result.plan.estimator}")
    print(f"Lags:        {result.plan.lags}")
    print(f"Metric:      {result.plan.metric}")
    print(f"Task type:   {result.plan.task_type}")
else:
    print("No structured plan returned (explanation-only response).")

No structured plan returned (explanation-only response).


### 4.2 Question with data context

When you pass `data`, the agent can use `profile_data` to inspect your
actual dataset and give a more precise recommendation.

In [ ]:
result_with_data = assistant.ask(
    question="Recommend a forecasting approach for this sales data with a 14-day steps.",
    data=df,
)

print(result_with_data.explanation)
print()
if result_with_data.plan:
    print("--- Plan ---")
    print(result_with_data.plan.model_dump_json(indent=2))

[LLM unavailable] A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['promo', 'temperature'].

--- Plan ---
{
  "task_type": "single_series",
  "forecaster": "ForecasterRecursive",
  "estimator": "Ridge",
  "horizon": 10,
  "frequency": "D",
  "lags": [
    1,
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "metric": "mean_absolute_error",
  "backtesting_strategy": "TimeSeriesFold",
  "interval_method": "bootstrapping",
  "dropna_from_series": false,
  "use_exog": true,
  "data_requirements": [],
  "warnings": [],
  "rationale": "A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['promo', 'temperature']."
}


/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_82258/4192376790.py:1: UserWarning: LLM call failed (status_code: 400, model_name: qwen3:8b, body: {'message': 'invalid message content type: <nil>', 'type': 'invalid_request_error', 'param': None, 'code': None}), falling back to deterministic mode.
  result_with_data = assistant.ask(


### 4.3 Selecting specific skills

You can control which domain knowledge the agent has access to by
specifying skill names. This reduces the system prompt size and
focuses the agent on relevant topics.

In [7]:
result_skills = assistant.ask(
    question="How should I set up prediction intervals for my daily sales forecast?",
    skills=["prediction-intervals", "forecasting-single-series"],
)

print(result_skills.explanation)

[LLM unavailable] Exceeded maximum retries (2) for output validation. Provide data to get a deterministic recommendation.


/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_82258/2867249802.py:1: UserWarning: LLM call failed (Exceeded maximum retries (2) for output validation), falling back to deterministic mode.
  result_skills = assistant.ask(


## 5. `explain()` — Plain-language plan explanations

The `explain()` method takes a `ForecastPlan` (and optionally a `DataProfile`)
and asks the LLM to produce a clear, non-technical explanation of why each
choice was made.

In [ ]:
# First, get a plan using the deterministic engine (no LLM needed)
rec = assistant.recommend(
    data=df,
    target="sales",
    date_column="date",
    steps=14,
)

print("Plan:")
print(f"  Forecaster: {rec.plan.forecaster}")
print(f"  Estimator:  {rec.plan.estimator}")
print(f"  Lags:       {rec.plan.lags}")
print(f"  Metric:     {rec.plan.metric}")
print(f"  Rationale:  {rec.plan.rationale}")

Plan:
  Forecaster: ForecasterRecursive
  Estimator:  Ridge
  Lags:       [1, 2, 3, 4, 5, 6, 7]
  Metric:     mean_absolute_error
  Rationale:  A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['promo', 'temperature'].


In [9]:
# Now ask the LLM to explain the plan
explanation = assistant.explain(
    plan=rec.plan,
    profile=rec.profile,
)

print(explanation)

Here’s a simple breakdown of the forecasting plan:  

1. **Goal**: Predict the next 14 days of values using historical data and external factors like promo and temperature.  
2. **Model Choice**:  
   - **Recursive Forecaster**: Uses past data (up to 7 days ago) to iteratively predict the next day, then uses those predictions as new data for the next step. This handles time dependencies well.  
   - **Ridge Regression**: A linear model that balances simplicity and accuracy, avoiding overfitting by penalizing extreme coefficients.  
3. **Data Use**:  
   - **Lags 1–7**: The model looks at the past 7 days to capture short-term patterns (e.g., daily trends or weekly cycles).  
   - **Exogenous Variables**: Promo (e.g., sales events) and temperature are included to explain how external factors might affect the forecast.  
4. **Evaluation**:  
   - **MAE**: Measures average prediction error in raw units (e.g., sales volume), making it easy to interpret.  
   - **TimeSeriesFold**: Tests the 

## 6. Graceful fallback when LLM is unavailable

If the LLM call fails (network error, wrong API key, model not found),
`ask()` falls back to the deterministic engine when data is provided,
and `explain()` returns the plan's built-in rationale.

In [10]:
# Simulate a broken LLM connection
broken_assistant = ForecastingAssistant(llm="ollama:nonexistent-model-xyz")

# ask() with data → falls back to deterministic recommend()
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    fallback_result = broken_assistant.ask(
        question="What model should I use?",
        data=df,
    )
    if w:
        print(f"Warning: {w[0].message}")

print(f"\nExplanation: {fallback_result.explanation}")
if fallback_result.plan:
    print(f"Fallback plan forecaster: {fallback_result.plan.forecaster}")


Explanation: [LLM unavailable] A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['promo', 'temperature'].
Fallback plan forecaster: ForecasterRecursive


## 7. Tier 0 vs LLM comparison

The LLM methods add a natural-language interface on top of the same
deterministic engine. Here's the full picture:

| Method | What it does | LLM required |
|---|---|---|
| `profile()` | Profile the dataset | No |
| `recommend()` | Profile → deterministic plan | No |
| `generate_code()` | Profile → plan → Python script | No |
| `forecast()` | Profile → plan → execute → predictions | No |
| **`ask()`** | **Natural-language → agent → tools → plan** | **Yes** |
| **`explain()`** | **Plan → plain-language explanation** | **Yes** |

The LLM never makes forecasting decisions. All recommendations come from
the deterministic rule engine. The LLM only translates intent and explains results.

## 8. Using a custom base URL

If you run Ollama on a different host or port, or use an OpenAI-compatible
endpoint, pass `base_url`:

In [11]:
# Example: Ollama on a remote server
# remote_assistant = ForecastingAssistant(
#     llm="ollama:qwen2.5:7b-instruct",
#     base_url="http://192.168.1.100:11434/v1",
# )
# result = remote_assistant.ask("What model for hourly electricity data?")
# print(result.explanation)